## 5 Обучение и выбор конфигурации

**Входы**

- Train, validation, шаблон предобработки и заранее выбранный протокол.

**Выходы**

- Обученный на train Pipeline-победитель.
- Зафиксированный по validation порог и таблица сравнения.

**Что делаем**

Обучаем восемь кандидатов с одинаковым набором исходных признаков. Сравниваем AP на validation; подбираем порог победителя. Test не участвует.

In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


#### Заранее заданная сетка

| Семейство | Конфигурации |
| --- | --- |
| Dummy | Константные вероятности по долям классов train |
| LogisticRegression | C=0.1; C=1; C=1 с class_weight=balanced; max_iter=2000 |
| RandomForest | 200 деревьев, min_samples_leaf=2; min_samples_leaf=5 с class_weight=balanced |
| HistGradientBoosting | 200 итераций, learning_rate=0.1, l2=1; max_leaf_nodes=15 или 31; early_stopping=False |

random_state=42 там, где он поддерживается; для леса n_jobs=1. Глубина леса не ограничена, число признаков разбиения — sqrt. Остальные параметры — значения по умолчанию зафиксированной версии scikit-learn.

Каждый кандидат получает **новый** преобразователь. Предобработка обучается вместе с классификатором через Pipeline.fit(X_train, y_train). Увеличение веса класса сравниваем как вариант, синтетические наблюдения не генерируем. Повторного поиска после просмотра test не будет.

In [ ]:
exec(compile(__import__('pathlib').Path('src/notebooks/_code/selection.py').read_text(encoding='utf-8'), 'src/notebooks/_code/selection.py', 'exec'), globals())

In [ ]:
exec(compile(__import__('pathlib').Path('src/notebooks/_code/train_models.py').read_text(encoding='utf-8'), 'src/notebooks/_code/train_models.py', 'exec'), globals())

AP и ROC-AUC оценивают ранжирование и не зависят от порога; Precision, Recall, F1, F2 и Accuracy зависят. Brier — средний квадрат ошибки вероятностного прогноза, меньше лучше. Таблица с порогом 0.5 показывает исходное поведение; в окончательное решение входит порог, выбранный по validation.

Validation используется и для выбора модели, и для порога, поэтому её метрики могут быть оптимистичны. Итоговое качество сообщаем по отложенному test. Устойчивость выбора модели к другим разбиениям данных не оценивалась.